# ForgeHLS kernel-shift QoR study: full reproduction

Run every cell top to bottom in one session. All tables and figures in the paper
must come from a single run of this notebook; mixing result files from different
runs or library versions gives inconsistent numbers.

Order: audit → build features → splits → baselines → tuned models → ablation
(E2 and five E3 seeds) → uncertainty → calibration → extra analyses → Pareto DSE proxy → deep-ensemble seed check → figures → export.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/VLSI CODE")
PROJECT_DRIVE = DRIVE_ROOT / "ForgeHLS"
DATASET_DRIVE = DRIVE_ROOT / "dataset" / "data_of_designs_forgehls.json"

print("Project exists:", PROJECT_DRIVE.exists(), PROJECT_DRIVE)
print("Dataset exists:", DATASET_DRIVE.exists(), DATASET_DRIVE)

if not PROJECT_DRIVE.exists():
    raise FileNotFoundError(f"ForgeHLS folder not found: {PROJECT_DRIVE}")

if not DATASET_DRIVE.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {DATASET_DRIVE}\n"
        "Check whether the dataset is inside MyDrive/VLSI CODE/dataset/"
    )

print("Dataset size GB:", round(DATASET_DRIVE.stat().st_size / 1024**3, 3))

In [ ]:
!rm -rf /content/ForgeHLS
!mkdir -p /content/ForgeHLS

In [ ]:
!cp -r "/content/drive/MyDrive/VLSI CODE/ForgeHLS/hls_ml" \
       "/content/ForgeHLS/hls_ml"

!cp "/content/drive/MyDrive/VLSI CODE/ForgeHLS/requirements.txt" \
    "/content/ForgeHLS/requirements.txt"

!cp "/content/drive/MyDrive/VLSI CODE/ForgeHLS/requirements-colab.txt" \
    "/content/ForgeHLS/requirements-colab.txt"

In [ ]:
%cd /content/ForgeHLS/hls_ml

In [ ]:
!pip install -q -r ../requirements.txt
!pip install -q -r ../requirements-colab.txt

In [ ]:
import sys
import numpy
import pandas
import scipy
import sklearn
import torch
import pyarrow

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("PyArrow:", pyarrow.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable. Deep ensemble will run very slowly on CPU.")

In [ ]:
import os
from pathlib import Path

DATASET_PATH = Path(
    "/content/drive/MyDrive/VLSI CODE/dataset/data_of_designs_forgehls.json"
)

if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

os.environ["FORGEHLS_DATA_PATH"] = str(DATASET_PATH)

print("FORGEHLS_DATA_PATH =", os.environ["FORGEHLS_DATA_PATH"])

In [ ]:
from pathlib import Path

SCRIPTS = Path("/content/ForgeHLS/hls_ml/scripts")

required_scripts = [
    "config.py",
    "feature_extractor.py",
    "dataset_audit.py",
    "build_dataset.py",
    "make_splits.py",
    "train_baselines.py",
    "train_tuned.py",
    "train_uncertainty.py",
    "evaluate.py",
    "ablation_study.py",
    "make_figures.py",
    "extra_analysis.py",
    "make_paper_figures.py",
    "pareto_dse.py",
    "deep_ensemble_stability.py",
    "row_order_audit.py",
]

for name in required_scripts:
    path = SCRIPTS / name
    print(name, "OK" if path.exists() else "MISSING")

if not all((SCRIPTS / name).exists() for name in required_scripts):
    raise FileNotFoundError("One or more required scripts are missing.")

feature_text = (SCRIPTS / "feature_extractor.py").read_text()
baseline_text = (SCRIPTS / "train_baselines.py").read_text()
uq_text = (SCRIPTS / "train_uncertainty.py").read_text()

print("Strict required-field validation:",
      "_required_float" in feature_text)

print("Log-MAE present:",
      "LogMAE" in baseline_text)

print("Stale Deep Ensemble reuse blocked:",
      "Refusing to reuse stale Deep Ensemble artifacts" in uq_text)

In [ ]:
!python -m py_compile scripts/*.py
!python scripts/code_review_checker.py

In [ ]:
!python scripts/test_feature_extractor.py

In [ ]:
!python scripts/dataset_audit.py

In [ ]:
!python scripts/build_dataset.py

In [ ]:
!python scripts/make_splits.py

In [ ]:
import numpy as np
import json

data = np.load("data/dataset_arrays.npz", allow_pickle=False)
splits = np.load("data/splits.npz", allow_pickle=False)

print("X shape:", data["X_features"].shape)
print("Y shape:", data["y_targets"].shape)
print("Groups:", len(np.unique(data["group_ids"])))
print("Feature count:", len(data["feature_names"]))
print("Target count:", len(data["target_names"]))

assert data["X_features"].shape == (184717, 42)
assert data["y_targets"].shape == (184717, 5)
assert len(np.unique(data["group_ids"])) == 491

for seed in [42, 43, 44, 45, 46]:
    train_key = f"e3_grouped_seed_{seed}_train_idx"
    test_key = f"e3_grouped_seed_{seed}_test_idx"
    assert train_key in splits
    assert test_key in splits

print("Fresh dataset and split verification PASSED")

In [ ]:
from pathlib import Path

RESULTS = Path("results")

old_outputs = [
    "baseline_results_e1_vs_e2.csv",
    "baseline_results_e3_repeated.csv",
    "baseline_e2_predictions.npz",
    "baseline_summary.txt",
    "uncertainty_predictions.npz",
    "uncertainty_results.csv",
    "uncertainty_summary.txt",
    "calibration_metrics.csv",
    "risk_coverage_data.csv",
    "calibration_selective_summary.txt",
    "feature_ablation_results.csv",
    "ablation_summary.txt",
    "tuned_model_selection.csv",
    "tuned_results_e2.csv",
    "tuned_results_e3_by_seed.csv",
    "tuned_results_e3_summary.csv",
    "feature_ablation_e3_by_seed.csv",
    "feature_ablation_e3_summary.csv",
    "extra_dataset_facts.csv",
    "extra_e1_e3_by_seed.csv",
    "extra_e1_e3_summary.csv",
    "extra_loso_by_source.csv",
    "extra_group_definitions.csv",
    "extra_uncertainty_magnitude.csv",
    "extra_uncertainty_quintiles.csv",
    "extra_risk_coverage_curves.csv",
    "extra_risk_coverage_aurc.csv",
    "extra_analysis_summary.txt",
    "pareto_dse_by_kernel.csv",
    "pareto_dse_by_seed.csv",
    "pareto_dse_summary.csv",
    "pareto_dse_summary.txt",
    "pareto_dse_heuristic_features.csv",
    "deep_ensemble_stability.csv",
    "row_order_audit.txt",
    "pareto_dse_by_kernel_row_ordered.csv",
    "pareto_dse_by_seed_row_ordered.csv",
    "pareto_dse_summary_row_ordered.csv",
    "pareto_dse_summary_row_ordered.txt",
    "pareto_dse_heuristic_features_row_ordered.csv",
]

for name in old_outputs:
    path = RESULTS / name
    if path.exists():
        path.unlink()
        print("Deleted:", path)

print("Old experiment artifacts removed.")

In [ ]:
# Record the exact library versions of this run next to the results.
# Tree models, HistGradientBoosting and the MLP ensemble change slightly
# across scikit-learn / PyTorch versions, so every reported number should
# come from one run in one environment.
!pip freeze > results/environment_freeze.txt
!python --version >> results/environment_freeze.txt
print(open("results/environment_freeze.txt").read()[-400:])


In [ ]:
!python scripts/train_baselines.py

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

baseline_csv = Path("results/baseline_results_e1_vs_e2.csv")
baseline_summary = Path("results/baseline_summary.txt")
baseline_predictions = Path("results/baseline_e2_predictions.npz")

assert baseline_csv.exists()
assert baseline_summary.exists()
assert baseline_predictions.exists()

baseline = pd.read_csv(baseline_csv)

required_columns = [
    "Model",
    "Target",
    "E1_R2",
    "E2_R2",
    "E1_LogMAE",
    "E2_LogMAE",
    "E1_LogR2",
    "E2_LogR2",
]

missing = [c for c in required_columns if c not in baseline.columns]
if missing:
    raise RuntimeError(f"Missing expected columns: {missing}")

numeric_columns = [
    "E1_R2",
    "E2_R2",
    "E1_MAE",
    "E2_MAE",
    "E1_LogMAE",
    "E2_LogMAE",
]

assert np.isfinite(baseline[numeric_columns].to_numpy()).all()

print("Baseline rows:", len(baseline))
print("Models:", baseline["Model"].unique())
print("Targets:", baseline["Target"].unique())
print("Fresh baseline verification PASSED")

display(baseline)

In [ ]:
for model, group in baseline.groupby("Model"):
    log_degradation = (group["E2_LogMAE"] > group["E1_LogMAE"]).sum()
    r2_degradation = (group["E1_R2"] > group["E2_R2"]).sum()

    print(
        f"{model}: "
        f"log-MAE degradation {log_degradation}/{len(group)}, "
        f"R2 degradation {r2_degradation}/{len(group)}"
    )

overall_log_degradation = (
    baseline["E2_LogMAE"] > baseline["E1_LogMAE"]
).mean()

print("Overall log-MAE degradation fraction:",
      round(float(overall_log_degradation), 3))

In [ ]:
!python scripts/train_tuned.py

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

tuned_files = [
    "results/tuned_model_selection.csv",
    "results/tuned_results_e2.csv",
    "results/tuned_results_e3_by_seed.csv",
    "results/tuned_results_e3_summary.csv",
]

for name in tuned_files:
    path = Path(name)
    print(name, "OK" if path.exists() else "MISSING")
    assert path.exists()

tuned_e2 = pd.read_csv("results/tuned_results_e2.csv")
tuned_e3 = pd.read_csv("results/tuned_results_e3_summary.csv")
assert len(tuned_e2) > 0
assert len(tuned_e3) > 0
required_e2_metrics = ["R2", "MAE", "MedianAE", "LogMAE"]
missing_e2_metrics = [c for c in required_e2_metrics if c not in tuned_e2.columns]
assert not missing_e2_metrics, f"Missing required E2 metrics: {missing_e2_metrics}"
assert np.isfinite(tuned_e2[required_e2_metrics].to_numpy()).all()

# Selection_LogMAE and Positive_F1 are optional for baseline/median rows.
optional_e2_metrics = ["Selection_LogMAE", "Positive_F1"]
present_optional_e2 = [c for c in optional_e2_metrics if c in tuned_e2.columns]
if present_optional_e2:
    assert not np.isinf(tuned_e2[present_optional_e2].to_numpy()).any()

e3_numeric = tuned_e3.select_dtypes(include="number")
assert np.isfinite(e3_numeric.to_numpy()).all()

print("Tuned E2 rows:", len(tuned_e2))
print("Tuned E3 rows:", len(tuned_e3))
print("Fresh tuned-model verification PASSED")
display(tuned_e3)


In [ ]:
!python scripts/ablation_study.py

In [ ]:
import pandas as pd

ablation = pd.read_csv("results/feature_ablation_results.csv")

print("Ablation rows:", len(ablation))
print("Feature sets:", ablation["Feature_Set"].unique())
print("Targets:", ablation["Target"].unique())

display(ablation)

In [ ]:
!python scripts/train_uncertainty.py

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

uq_files = [
    "results/uncertainty_predictions.npz",
    "results/uncertainty_results.csv",
    "results/uncertainty_summary.txt",
]

for name in uq_files:
    path = Path(name)
    print(name, "OK" if path.exists() else "MISSING")
    assert path.exists()

uq = pd.read_csv("results/uncertainty_results.csv")
pred = np.load("results/uncertainty_predictions.npz", allow_pickle=False)

print("UQ rows:", len(uq))
print("UQ methods:", uq["Method"].unique())
print("Prediction arrays:", pred.files)

assert len(uq) == 10
print("Fresh uncertainty verification PASSED")

display(uq)

In [ ]:
!python scripts/evaluate.py

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

calibration = pd.read_csv("results/calibration_metrics.csv")
risk_coverage = pd.read_csv("results/risk_coverage_data.csv")

assert Path("results/calibration_selective_summary.txt").exists()
assert len(calibration) == 5
assert len(risk_coverage) > 0

print("Calibration rows:", len(calibration))
print("Risk-coverage rows:", len(risk_coverage))

assert np.isfinite(calibration.select_dtypes("number").to_numpy()).all()
assert np.isfinite(risk_coverage.select_dtypes("number").to_numpy()).all()

display(calibration)
display(risk_coverage.head(20))

In [ ]:
# Paper-only analyses: constant baseline, five-seed E1, within-kernel ranking,
# unroll heuristic, algorithm-name grouping, leave-one-source-out, magnitude
# baseline, decile/resampling checks, risk-coverage areas, feature collisions.
!python scripts/extra_analysis.py


In [ ]:
# Budgeted DSE proxy on held-out kernels: Pareto recall and ADRS for the
# model, a pragma heuristic, random choice and an oracle.
!python scripts/pareto_dse.py
# Diagnostic: ForgeHLS lists Pareto variants early within each kernel, so
# index-based tie breaking leaks the label. Show the size of that effect.
!python scripts/row_order_audit.py
!python scripts/pareto_dse.py --keep-row-order


In [ ]:
# Deep-ensemble seed stability (three disjoint seed sets).
!python scripts/deep_ensemble_stability.py


In [ ]:
!python scripts/make_figures.py

In [ ]:
# Figures exactly as used in the paper (Fig. 1-4).
!python scripts/make_paper_figures.py


In [ ]:
from IPython.display import Image, display
from pathlib import Path

figure_names = [
    "fig1_distribution_shift_e1_vs_e2.png",
    "fig2_uncertainty_correlation_quintiles.png",
    "fig3_risk_coverage_curves.png",
    "fig4_feature_ablation.png",
    "paper_fig1_e1_vs_e3_logmae.png",
    "paper_fig2_uncertainty_quintiles.png",
    "paper_fig3_risk_coverage.png",
    "paper_fig4_feature_ablation.png",
]

for name in figure_names:
    path = Path("figures") / name
    print(name, "OK" if path.exists() else "MISSING")
    assert path.exists()
    display(Image(filename=str(path)))

In [ ]:
import os
import pandas as pd

os.makedirs("tables", exist_ok=True)

files = {
    "baseline": "results/baseline_results_e1_vs_e2.csv",
    "repeated": "results/baseline_results_e3_repeated.csv",
    "uncertainty": "results/uncertainty_results.csv",
    "calibration": "results/calibration_metrics.csv",
    "risk_coverage": "results/risk_coverage_data.csv",
    "ablation": "results/feature_ablation_results.csv",
    "tuned_e2": "results/tuned_results_e2.csv",
    "tuned_e3_summary": "results/tuned_results_e3_summary.csv",
    "ablation_e3_summary": "results/feature_ablation_e3_summary.csv",
    "extra_dataset_facts": "results/extra_dataset_facts.csv",
    "extra_e1_e3_summary": "results/extra_e1_e3_summary.csv",
    "extra_group_definitions": "results/extra_group_definitions.csv",
    "extra_loso_by_source": "results/extra_loso_by_source.csv",
    "extra_uncertainty_magnitude": "results/extra_uncertainty_magnitude.csv",
    "extra_risk_coverage_aurc": "results/extra_risk_coverage_aurc.csv",
    "pareto_dse_summary": "results/pareto_dse_summary.csv",
    "deep_ensemble_stability": "results/deep_ensemble_stability.csv",
}

for name, path in files.items():
    df = pd.read_csv(path)

    df.to_csv(f"tables/{name}.csv", index=False)

    with open(f"tables/{name}.tex", "w", encoding="utf-8") as output:
        output.write(
            df.to_latex(
                index=False,
                float_format="%.4f"
            )
        )

    print("Exported:", name)

print("All tables exported.")

In [ ]:
from pathlib import Path
import pandas as pd

print("=== BASELINE SUMMARY ===")
print(Path("results/baseline_summary.txt").read_text())

print("\n=== UNCERTAINTY SUMMARY ===")
print(Path("results/uncertainty_summary.txt").read_text())

print("\n=== CALIBRATION SUMMARY ===")
print(Path("results/calibration_selective_summary.txt").read_text())

print("\n=== RESULT FILES ===")
for path in sorted(Path("results").glob("*")):
    print(path, round(path.stat().st_size / 1024, 1), "KB")

In [ ]:
from pathlib import Path
import shutil

DRIVE_HLS = Path("/content/drive/MyDrive/VLSI CODE/ForgeHLS/hls_ml")

for folder in ["data", "results", "figures", "tables"]:
    source = Path(folder)
    destination = DRIVE_HLS / folder
    destination.mkdir(parents=True, exist_ok=True)

    for item in source.iterdir():
        target = destination / item.name

        if item.is_file():
            shutil.copy2(item, target)
        elif item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)

    print("Synced:", folder)

print("Fresh data, results, figures, and tables copied to Google Drive.")

In [ ]:
from pathlib import Path
from datetime import datetime

for folder in ["results", "figures", "tables"]:
    print(f"\n--- {folder} ---")

    for path in sorted(Path(folder).glob("*")):
        modified = datetime.fromtimestamp(path.stat().st_mtime)
        print(modified.strftime("%Y-%m-%d %H:%M:%S"), path.name)